In [3]:
!apt-get update --fix-missing
!apt-get -qq install -y libquantlib0-dev
!pip -q install QuantLib tqdm

## Import custom modules from github
!rm -rf deep-hedging
!git clone https://github.com/YuMan-Tam/deep-hedging

import sys, os
sys.path.insert(0, os.getcwd() + "/deep-hedging")
!pip install tensorflow==1.15.0
# The kernel needs to be restarted after installing tensorflow
# Uncomment the following line to restart the kernel automatically
!restart_kernel

import tensorflow as tf # This line should work after kernel restart
print(tf.__version__)

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Ign:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy Release
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Cloning into

ModuleNotFoundError: No module named 'tensorflow'

In [2]:

import tensorflow as tf # This line should work after kernel restart
print(tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'

In [3]:
!pip install tensorflow==1.15.0 # Ensure tensorflow is installed
import tensorflow as tf # This line should work after kernel restart
print(tf.__version__)

ERROR: Could not find a version that satisfies the requirement tensorflow==1.15.0 (from versions: 2.8.0rc0, 2.8.0rc1, 2.8.0, 2.8.1, 2.8.2, 2.8.3, 2.8.4, 2.9.0rc0, 2.9.0rc1, 2.9.0rc2, 2.9.0, 2.9.1, 2.9.2, 2.9.3, 2.10.0rc0, 2.10.0rc1, 2.10.0rc2, 2.10.0rc3, 2.10.0, 2.10.1, 2.11.0rc0, 2.11.0rc1, 2.11.0rc2, 2.11.0, 2.11.1, 2.12.0rc0, 2.12.0rc1, 2.12.0, 2.12.1, 2.13.0rc0, 2.13.0rc1, 2.13.0rc2, 2.13.0, 2.13.1, 2.14.0rc0, 2.14.0rc1, 2.14.0, 2.14.1, 2.15.0rc0, 2.15.0rc1, 2.15.0, 2.15.0.post1, 2.15.1, 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.18.0rc0)
ERROR: No matching distribution found for tensorflow==1.15.0


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:

from IPython.display import clear_output

import numpy as np
import QuantLib as ql
import tensorflow as tf
from scipy.stats import norm

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, \
                                            ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model

import matplotlib.pyplot as plt

from stochastic_processes import BlackScholesProcess
from instruments import EuropeanCall
from deep_hedging import Deep_Hedging_Model, Delta_SubModel
from loss_metrics import Entropy
from utilities import train_test_split
from keras.layers import Layer
import tensorflow as tf
clear_output()

In [ ]:
print(tf.__version__)

# Reference: Deep Hedging (2019, Quantitative Finance) by Buehler et al.
# https://www.tandfonline.com/doi/abs/10.1080/14697688.2019.1571683


# ** USER input**
#Provide input parameters for Monte Carlo simulation, call option, transaction cost
#loss function, and deep hedging algorithm algo

In [ ]:
# Geometric Brownian Motion.
N = 30 # Number of time steps (in days)

S0 = 100.0 # Stock price at time = 0
sigma = 0.2 # Implied volatility
risk_free = 0.0 # Risk-free rate
dividend = 0.0 # Continuous dividend yield

Ktrain = 1*(10**5) # Size of training sample.
Ktest_ratio = 0.2 # Fraction of training sample as testing sample.

# European call option (short).
strike = S0
payoff_func = lambda x: -np.maximum(x - strike, 0.0)
calculation_date = ql.Date.todaysDate()
maturity_date = ql.Date.todaysDate() + N

# Day convention.
day_count = ql.Actual365Fixed() # Actual/Actual (ISDA)

# Proportional transaction cost.
epsilon = 0.0

# Information set (in string)
# Choose from: S, log_S, normalized_log_S (by S0)
information_set = "normalized_log_S"

# Loss function
# loss_type = "CVaR" (Expected Shortfall) -> loss_param = alpha
# loss_type = "Entropy" -> loss_param = lambda

loss_type = "Entropy"
loss_param = 1.0

# Neural network (NN) structure
m = 15 # Number of neurons in each hidden layer.
d = 1 # Number of hidden layers (Note including input nor output layer)

# Neural network training parameters
lr = 1e-2 # Learning rate
batch_size=256 # Batch size
epochs=50 # Number of epochs

# Other parameters
use_batch_norm = False
kernel_initializer = "he_uniform"

activation_dense = "leaky_relu"
activation_output = "sigmoid"
final_period_cost = False

delta_constraint = (0.0, 1.0)
share_stretegy_across_time = False
cost_structure = "proportional"

# Other control flags for development purpose.
mc_simulator = "QuantLib" # "QuantLib" or "Numpy"

# **Monte Carlo Simulation - Generate RandomPaths of Stock Prices１**


In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# # Parameters
# S0 = 100  # Initial stock price
# V0 = 0.04  # Initial variance
# T = 1.0  # Time horizon (1 year)
# N = 252  # Number of time steps (trading days in a year)
# dt = T / N  # Time step
# mu = 0.05  # Drift (annual return)
# kappa = 2.0  # Mean reversion rate
# theta = 0.04  # Long-term variance
# sigma = 0.2  # Volatility of volatility
# rho = -0.7  # Correlation between the two Brownian motions
# num_simulations = 1000  # Number of simulations

# # Generate random paths
# np.random.seed(42)  # For reproducibility
# S = np.zeros((num_simulations, N + 1))
# V = np.zeros((num_simulations, N + 1))
# S[:, 0] = S0
# V[:, 0] = V0

# for t in range(1, N + 1):
#     z1 = np.random.standard_normal(num_simulations)
#     z2 = np.random.standard_normal(num_simulations)
#     W1 = z1 * np.sqrt(dt)
#     W2 = rho * z1 * np.sqrt(dt) + np.sqrt(1 - rho**2) * z2 * np.sqrt(dt)

#     V[:, t] = np.maximum(V[:, t-1] + kappa * (theta - V[:, t-1]) * dt + sigma * np.sqrt(V[:, t-1]) * W2, 0)
#     S[:, t] = S[:, t-1] * np.exp((mu - 0.5 * V[:, t-1]) * dt + np.sqrt(V[:, t-1]) * W1)

# # Plotting the paths
# plt.figure(figsize=(10, 6))
# for i in range(num_simulations):
#     plt.plot(S[i], lw=0.5)
# plt.title('Heston Model Simulation of Stock Prices')
# plt.xlabel('Time Steps')
# plt.ylabel('Stock Price')
# plt.show()


# **Monte Carlo Simulation - Generate RandomPaths of Stock Prices2**

In [ ]:
seed = 0 # Random seed. Change to have deterministic outcome.

# Total obs = Training + Testing
nobs = int(Ktrain*(1+Ktest_ratio))

# Length of one time-step (as fraction of a year).
dt = day_count.yearFraction(calculation_date,calculation_date + 1)
maturity = N*dt # Maturities (in the unit of a year)

stochastic_process = BlackScholesProcess(s0 = S0, sigma = sigma, risk_free = risk_free, \
                        dividend = dividend, day_count = day_count, seed=seed)

S = stochastic_process.gen_path(maturity, N, nobs)

clear_output()

print("\n\ns0 = " + str(S0))
print("sigma = " + str(sigma))
print("risk_free = " + str(risk_free) + "\n")
print("Number of time steps = " + str(N))
print("Length of each time step = " + "1/365\n")
print("Simulation Done!")

# **Prepare data to be fed into the deep hedging algorithm.**

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Assuming S is defined and payoff_func is available
payoff_T = payoff_func(S[:, -1])  # Payoff of the call option
trade_set = np.stack((S), axis=1)  # Trading set

# Determine the information set based on the input
if information_set == "S":
    I = np.stack((S), axis=1)  # Information set
elif information_set == "log_S":
    I = np.stack((np.log(S)), axis=1)
elif information_set == "normalized_log_S":
    I = np.stack((np.log(S / S0)), axis=1)

# Structure of xtrain
#   1) Trade set: [S]
#   2) Information set: [S]
#   3) Payoff (dim = 1)
x_all = []
for i in range(N + 1):
    x_all.append(trade_set[i, :, None])  # Trade set
    if i != N:
        x_all.append(I[i, :, None])  # Information set
x_all.append(payoff_T[:, None])  # Payoff as the last element

# Create x (features) and y (labels)
x_features = np.concatenate(x_all[:-1], axis=-1)  # All but the last element (payoff)
y_labels = x_all[-1]  # The last element (payoff)

# Split the entire sample into a training sample and a testing sample.
xtrain, xtest, ytrain, ytest = train_test_split(x_features, y_labels, test_size=int(Ktrain * Ktest_ratio))

# Ensure the data shapes are consistent
print("xtrain shape:", np.shape(xtrain))
print("ytrain shape:", np.shape(ytrain))
print("xtest shape:", np.shape(xtest))
print("ytest shape:", np.shape(ytest))

print("Finish preparing data!")


# **Run the Deep Hedging Algorithm (Simple Network)!**

In [ ]:
optimizer = Adam(learning_rate=lr)

# Setup and compile the model
model_recurrent = Deep_Hedging_Model(N=N, d=d+2, m=m, risk_free=risk_free, \
          dt = dt, strategy_type="recurrent", epsilon = epsilon, \
          use_batch_norm = use_batch_norm, kernel_initializer = kernel_initializer, \
          activation_dense = activation_dense, activation_output = activation_output, \
          final_period_cost = final_period_cost)

loss = Entropy(model_recurrent.output,None,loss_param)
model_recurrent.add_loss(loss)

model_recurrent.compile(optimizer=optimizer)

early_stopping = EarlyStopping(monitor="loss", \
          patience=10, min_delta=1e-4, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="loss", \
          factor=0.5, patience=2, min_delta=1e-3, verbose=0)

callbacks = [early_stopping, reduce_lr]

# Fit the model.
model_recurrent.fit(x=xtrain, batch_size=batch_size, epochs=epochs, \
          validation_data=xtest, verbose=1)

clear_output()

In [ ]:
optimizer = Adam(learning_rate=lr)

# Setup and compile the model
model_simple = Deep_Hedging_Model(N=N, d=d+2, m=m, risk_free=risk_free, \
          dt = dt, strategy_type="simple", epsilon = epsilon, \
          use_batch_norm = use_batch_norm, kernel_initializer = kernel_initializer, \
          activation_dense = activation_dense, activation_output = activation_output, \
          final_period_cost = final_period_cost, delta_constraint = delta_constraint, \
          share_stretegy_across_time = share_stretegy_across_time, \
          cost_structure = cost_structure)
loss = Entropy(model_simple.output,None,loss_param)
model_simple.add_loss(loss)

model_simple.compile(optimizer=optimizer)

early_stopping = EarlyStopping(monitor="loss", \
          patience=10, min_delta=1e-4, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor="loss", \
          factor=0.5, patience=2, min_delta=1e-3, verbose=0)

callbacks = [early_stopping, reduce_lr]

# Fit the model.
model_simple.fit(x=xtrain, batch_size=batch_size, epochs=epochs, \
          validation_data=xtest, verbose=1)

clear_output()

print("Finished running deep hedging algorithm! (Simple Network)")